In [1]:
import pyarrow.dataset as ds
import pyarrow.fs as fs
import pyarrow.parquet as pq

In [2]:
filesystem = fs.HadoopFileSystem("hdfs://arnsdpsbx", port=0)

In [3]:
target_1_path = "/user/team/team_sbertype_evolution/beka_data/dlt/custom_heads/home/targets_test"
target_2_path = "/user/team/team_sbertype_evolution/beka_data/dlt/custom_heads/russ/targets_test"

In [4]:
remote_path_1 = "/user/team/team_sbertype_evolution/beka_data/dlt/custom_heads/home/dataset_test"
remote_path_2 = "/user/team/team_sbertype_evolution/beka_data/dlt/custom_heads/russ/dataset_test"

In [5]:
dataset_1 = ds.dataset(remote_path_1, filesystem=filesystem, format="parquet")
dataset_2 = ds.dataset(remote_path_2, filesystem=filesystem, format="parquet")

In [6]:
targets_1 = ds.dataset(target_1_path, filesystem=filesystem, format="parquet")
targets_2 = ds.dataset(target_2_path, filesystem=filesystem, format="parquet")

In [7]:
import polars as pl

In [8]:
pl_targets_1 = pl.from_arrow(targets_1.to_table()).sort("epk_id", "report_dt")
pl_targets_2 = pl.from_arrow(targets_2.to_table()).sort("epk_id", "report_dt")

In [9]:
pl_targets = pl_targets_1.join(pl_targets_2, on=("epk_id", "report_dt"), how="full", coalesce=True).sort("epk_id", "report_dt")

In [10]:
target_names = ["vacation_home_d30", "vacation_russia_d30"]

In [11]:
from tqdm.autonotebook import tqdm

In [12]:
names = ["dir_token", "ecom_token", "mcc_token", "brand_token", "city_token", "amt_token"]
all_names = ["date_stamp", *names]

In [13]:
def handle_batch(pl_batch: pl.DataFrame) -> pl.DataFrame:
    result: pl.DataFrame = (
        pl_batch.with_columns(pl.col("report_dt").dt.timestamp("ms").floordiv(1000))
        .with_columns(
            pl.lit([])
            .list.concat(
                [
                    pl.col("value").fill_null([]),
                    pl.col("vnv_value").fill_null([]),
                    pl.col("okko_value").fill_null([]),
                    pl.col("samokat_value").fill_null([]),
                ]
            )
            .alias("all_values"),
        )
        .select("epk_id", "report_dt", "all_values", *target_names)
        .with_columns(
            pl.col("all_values").list.eval(pl.element().list.get(j, null_on_oob=True).fill_null(0)).alias(name)
            for j, name in enumerate(all_names)
        )
        .select("epk_id", "report_dt", *target_names, *all_names)
        .explode(all_names)
        .sort("date_stamp", "epk_id", "report_dt", maintain_order=True)
        .group_by(
            ["epk_id", "report_dt", *target_names],
            maintain_order=True,
        )
        .agg(*all_names)
        .with_columns(pl.col("date_stamp").list.concat(pl.col("report_dt")))
        .with_columns((pl.col(name).list.concat(pl.lit(3)) for j, name in enumerate(names)))
        .select("epk_id", "report_dt", *target_names, *all_names)
        .sort("epk_id", "report_dt")
    )

    return result

In [14]:
!rm -r /home/datalab/nfs/romashka_test_data_multitarget
!mkdir -p /home/datalab/nfs/romashka_test_data_multitarget

In [15]:
for i, batch in enumerate(tqdm(dataset_1.to_batches())):
    pl_batch = pl.from_arrow(batch).join(pl_targets, on=("epk_id", "report_dt")).sort("epk_id", "report_dt")

    pl_result = handle_batch(pl_batch).drop_nulls((*all_names, "epk_id", "report_dt")).sort("epk_id", "report_dt")

    result_name: str = f"/home/datalab/nfs/romashka_test_data_multitarget/batch_tgt_1_{i:04d}.parquet"
    pq.write_table(pl_result.to_arrow(), result_name)

0it [00:00, ?it/s]

In [16]:
for j, batch in enumerate(tqdm(dataset_2.to_batches())):
    k = j + i
    pl_batch = pl.from_arrow(batch).join(pl_targets, on=("epk_id", "report_dt")).sort("epk_id", "report_dt")

    pl_result = handle_batch(pl_batch).drop_nulls((*all_names, "epk_id", "report_dt")).sort("epk_id", "report_dt")

    result_name: str = f"/home/datalab/nfs/romashka_test_data_multitarget/batch_tgt_2_{k:04d}.parquet"
    pq.write_table(pl_result.to_arrow(), result_name)

0it [00:00, ?it/s]